# 04 · Deploy both agents

Each agent deploys in two steps with two tools:

1. **`agents-cli infra single-project --apply`** runs Terraform: service accounts or Agent Identity, IAM,
   telemetry (Cloud Trace, logging buckets, BigQuery datasets) and, for Agent Runtime, the engine itself.
2. **`agents-cli deploy`** builds the container from your code and puts it on that infrastructure.

The Compliance Agent goes first, because the orchestrator calls it. Terraform state lives in the bucket from
setup step 1. Deploying is billable; the agents scale to zero when idle.

In [ ]:
from nbsetup import *

## The Compliance Agent on Cloud Run

### Infrastructure

`infra single-project` without `--apply` shows the Terraform plan: about 30 resources, including a private Cloud
Run service with a placeholder image, its own service account, telemetry, `roles/run.invoker` for the caller
service account and an Agent Registry entry.

In [ ]:
show("compliance-agent/deployment/terraform/single-project/custom_iam.tf", 'resource "google_cloud_run_v2_service_iam_member"', lang="hcl")

In [ ]:
%%bash
cd "$REPO/compliance-agent" && agents-cli infra single-project --project "$PROJECT" 2>&1 | grep -E "Plan:|Error|error" 

In [ ]:
%%bash
cd "$REPO/compliance-agent" && agents-cli infra single-project --project "$PROJECT" --apply 2>&1 | tail -5

### Code

`agents-cli deploy` builds the image with Cloud Build and deploys it to the Cloud Run service. The sizing flags
must match the Terraform (`service.tf`), or the next `terraform plan` shows drift. A few minutes.

In [ ]:
%%bash
cd "$REPO/compliance-agent"
agents-cli deploy --project "$PROJECT" --region "$REGION" --no-confirm-project \
  --service-account "compliance-agent-app@$PROJECT.iam.gserviceaccount.com" \
  --memory 2Gi --cpu 1 --min-instances 0 --max-instances 3 --concurrency 4 2>&1 | tail -8

### Talk to it over A2A

A2A (Agent2Agent) is how independently deployed agents talk: an **Agent Card** at
`/.well-known/agent-card.json` describes the agent, and calls are JSON-RPC `message/send`. The service is
private, so both calls carry an ID token minted by impersonating the caller service account.

In [ ]:
%%bash
echo "without a token: HTTP $(curl -s -o /dev/null -w '%{http_code}' "$COMPLIANCE_URL/a2a/app/.well-known/agent-card.json")"
TOKEN=$(gcloud auth print-identity-token --impersonate-service-account "$MCP_CALLER" --audiences "$COMPLIANCE_URL" --include-email 2>/dev/null)
curl -s -H "Authorization: Bearer $TOKEN" "$COMPLIANCE_URL/a2a/app/.well-known/agent-card.json" \
  | python3 -c "import json,sys; c=json.load(sys.stdin); print(c['name'], '-', c['description']); print('skills:', [s['name'] for s in c['skills']])"

In [ ]:
%%bash
TOKEN=$(gcloud auth print-identity-token --impersonate-service-account "$MCP_CALLER" --audiences "$COMPLIANCE_URL" --include-email 2>/dev/null)
curl -s -X POST "$COMPLIANCE_URL/a2a/app" -H "Authorization: Bearer $TOKEN" -H "Content-Type: application/json" -d '{
  "jsonrpc": "2.0", "id": "1", "method": "message/send",
  "params": {"message": {"role": "user", "messageId": "notebook-1", "kind": "message",
    "parts": [{"kind": "text", "text": "The user directly requested this action: approve_discount (campaign_id: CAMP-2025-099, percentage: 35, reason: boost conversions). Check exactly this action."}]}}
}' | python3 -c "
import json, sys
r = json.load(sys.stdin)['result']
print('task state:', r['status']['state'])
for a in r.get('artifacts', []):
    for p in a['parts']:
        if p.get('text'): print(p['text'])"

### Agent Registry

Agent Runtime agents are registered automatically (as type `CUSTOM`). Cloud Run agents aren't, so Terraform
registers the Compliance Agent from its card, as type **A2A**.

In [ ]:
show("compliance-agent/deployment/terraform/single-project/custom_registry.tf", 'resource "google_agent_registry_service"', lang="hcl")

In [ ]:
%%bash
gcloud alpha agent-registry agents list --location "$REGION" --format json \
  | python3 -c "import json,sys; [print(f\"{a['displayName']:28} {a['protocols'][0]['type']:10} skills={[s['name'] for s in (a.get('skills') or [])]}\") for a in json.load(sys.stdin)]"

## The orchestrator on Agent Runtime

### Infrastructure

On Agent Runtime, Terraform **creates the engine** and owns it; `agents-cli deploy` later only replaces the code.
The engine runs as its own **Agent Identity**, and Terraform grants that principal exactly the roles it needs,
plus permission to impersonate the caller service account.

In [ ]:
show("growth-risk-agent/deployment/terraform/single-project/service.tf", 'resource "google_vertex_ai_reasoning_engine" "app"', lines=40, lang="hcl")

In [ ]:
show("growth-risk-agent/deployment/terraform/single-project/custom_iam.tf", 'resource "google_service_account_iam_member"', lang="hcl")

**What to notice**

- `lifecycle.ignore_changes` lists the container, source, deployment spec and class methods: those belong to `agents-cli deploy`. Without it, the next `terraform apply` would undo the deploy.
- A consequence: env vars in `service.tf` only apply when the engine is created. Later changes go through `agents-cli deploy`, which reads them from `.env`.

In [ ]:
%%bash
cd "$REPO/growth-risk-agent" && agents-cli infra single-project --project "$PROJECT" 2>&1 | grep -E "Plan:|Error|error"

In [ ]:
%%bash
cd "$REPO/growth-risk-agent" && agents-cli infra single-project --project "$PROJECT" --apply 2>&1 | tail -5

### Code

A dry run first: it checks APIs, the target engine and IAM without changing anything. Then the real deploy, which
packages the code, builds the image and updates the engine Terraform created (it finds it by name). About 8
minutes. Always pass `--agent-identity`: without it, a *newly created* engine gets no Agent Identity.

In [ ]:
%%bash
cd "$REPO/growth-risk-agent" && agents-cli deploy --project "$PROJECT" --region "$REGION" --agent-identity --update-only --no-confirm-project --dry-run 2>&1 | tail -12

In [ ]:
%%bash
cd "$REPO/growth-risk-agent" && agents-cli deploy --project "$PROJECT" --region "$REGION" --agent-identity --update-only --no-confirm-project 2>&1 | tail -12

### Check what was deployed

`deployment_metadata.json` now names the engine. Its identity should be an Agent Identity principal, and every
value from `.env` should be in its environment.

In [ ]:
import google.auth, google.auth.transport.requests, requests

creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
creds.refresh(google.auth.transport.requests.Request())
e = requests.get(f"https://{REGION}-aiplatform.googleapis.com/v1beta1/{engine()}",
                 headers={"Authorization": f"Bearer {creds.token}"}).json()
spec = e["spec"]
print("engine   :", engine())
print("identity :", spec.get("identityType"), "→", spec.get("effectiveIdentity"))
print("sizing   :", {k: spec["deploymentSpec"].get(k) for k in ("minInstances", "maxInstances", "containerConcurrency", "resourceLimits")})
env = {v["name"]: v.get("value", "") for v in spec["deploymentSpec"].get("env", [])}
for key in ["BQ_DATASET", "POLICY_RAG_CORPUS", "SUPPORT_RAG_CORPUS", "REMEDIATION_MCP_URL",
            "MCP_CALLER_SERVICE_ACCOUNT", "COMPLIANCE_AGENT_URL", "GOOGLE_CLOUD_LOCATION"]:
    print(f"  {'✅' if env.get(key) else '❌'} {key} = {env.get(key, '(missing)')}")

Next: [05 · Use the deployed agent](05_use_the_deployed_agent.ipynb).